In [1]:
!pip install -q transformers datasets evaluate rouge-score sacrebleu faiss-cpu accelerate


In [2]:
!pip install -q transformers torch pandas tqdm


In [3]:
from torch.utils.data import DataLoader
import os
import json
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
import evaluate
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


In [4]:
optimizer = AdamW([torch.randn(2, requires_grad=True)], lr=2e-5)
print("AdamW OK")


AdamW OK


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [6]:
with open("/content/pubMed.json") as f:
    pubmedqa_data = json.load(f)

with open("/content/BioASQ12b.json") as f:
    bioasq_data = json.load(f)


In [7]:
class BiomedicalQADataset(Dataset):
    def __init__(self, data, dataset_type, tokenizer):
        self.samples = []
        self.tokenizer = tokenizer
        self.dataset_type = dataset_type

        if dataset_type == "pubmedqa":
            for _, item in data.items():
                question = item["QUESTION"]
                context = " ".join(item["CONTEXTS"])
                answer = item["LONG_ANSWER"]

                self.samples.append((question, context, answer))

        elif dataset_type == "bioasq":
            for q in data["questions"]:
                question = q["body"]
                context = " ".join([s["text"] for s in q["snippets"]])
                answer = " ".join(q["ideal_answer"])

                self.samples.append((question, context, answer))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        question, context, answer = self.samples[idx]

        encoder_input = self.tokenizer(
            question + " " + context,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        with self.tokenizer.as_target_tokenizer():
            decoder_target = self.tokenizer(
                answer,
                truncation=True,
                padding="max_length",
                max_length=128,
                return_tensors="pt"
            )

        return {
            "input_ids": encoder_input["input_ids"].squeeze(0),
            "attention_mask": encoder_input["attention_mask"].squeeze(0),
            "labels": decoder_target["input_ids"].squeeze(0)
        }


In [8]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

pubmed_dataset = BiomedicalQADataset(
    pubmedqa_data, "pubmedqa", tokenizer
)

bioasq_dataset = BiomedicalQADataset(
    bioasq_data, "bioasq", tokenizer
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
pubmed_dataset.samples = pubmed_dataset.samples[:500]
bioasq_dataset.samples = bioasq_dataset.samples[:500]


In [10]:
pubmed_loader = DataLoader(pubmed_dataset, batch_size=2, shuffle=True)
bioasq_loader = DataLoader(bioasq_dataset, batch_size=2, shuffle=True)


In [11]:
from transformers import BartForConditionalGeneration

class WhyMedQAModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
        self.attn = torch.nn.MultiheadAttention(768, 8, batch_first=True)
        self.fc = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.1)
        self.norm = torch.nn.LayerNorm(768)

    def forward(self, input_ids, attention_mask, labels):
        # ---- Let BART handle decoder + loss ----
        outputs = self.bart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,
            return_dict=True
        )

        # ---- Apply custom layers on decoder hidden states ----
        dec_out = outputs.decoder_hidden_states[-1]

        attn_out, _ = self.attn(dec_out, dec_out, dec_out)
        x = self.fc(attn_out)
        x = torch.nn.functional.gelu(x)
        x = self.dropout(x)
        x = self.norm(x + dec_out)

        # ---- (Optional) auxiliary loss / future use ----
        # logits = self.bart.lm_head(x)

        return outputs.loss


In [12]:
model = WhyMedQAModel().to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 10
total_steps = epochs * (len(pubmed_loader) + len(bioasq_loader))

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0.1 * total_steps,
    num_training_steps=total_steps
)


In [13]:
logs = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    steps = 0

    for loader, name in [(pubmed_loader, "PubMedQA"), (bioasq_loader, "BioASQ")]:
        for batch in tqdm(loader, desc=f"Epoch {epoch+1} | {name}"):
            batch = {k: v.to(device) for k, v in batch.items()}

            loss = model(**batch)
            loss.backward()

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()
            steps += 1

    avg_loss = total_loss / steps
    logs.append({"epoch": epoch + 1, "loss": avg_loss})
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")


Epoch 1 | PubMedQA:   0%|          | 0/250 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4174: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Epoch 1 | PubMedQA: 100%|██████████| 250/250 [00:57<00:00,  4.35it/s]
Epoch 1 | BioASQ: 100%|██████████| 250/250 [00:58<00:00,  4.26it/s]


Epoch 1 | Avg Loss: 4.3317


Epoch 2 | PubMedQA: 100%|██████████| 250/250 [00:53<00:00,  4.64it/s]
Epoch 2 | BioASQ: 100%|██████████| 250/250 [00:54<00:00,  4.56it/s]


Epoch 2 | Avg Loss: 1.1355


Epoch 3 | PubMedQA: 100%|██████████| 250/250 [00:53<00:00,  4.64it/s]
Epoch 3 | BioASQ: 100%|██████████| 250/250 [00:58<00:00,  4.25it/s]


...

Epoch 8 | Avg Loss: 0.7300


Epoch 9 | PubMedQA: 100%|██████████| 250/250 [00:53<00:00,  4.66it/s]
Epoch 9 | BioASQ: 100%|██████████| 250/250 [00:54<00:00,  4.58it/s]


Epoch 9 | Avg Loss: 0.6989


Epoch 10 | PubMedQA: 100%|██████████| 250/250 [00:53<00:00,  4.66it/s]
Epoch 10 | BioASQ: 100%|██████████| 250/250 [00:54<00:00,  4.57it/s]

Epoch 10 | Avg Loss: 0.6782


In [14]:
df = pd.DataFrame(logs)
df.to_csv("whymedqa_prvg_training_logs.csv", index=False)


In [15]:
torch.save(model.state_dict(), "whymedqa_prvg_model.pt")
tokenizer.save_pretrained("whymedqa_tokenizer")


NameError: name 'train_data' is not defined